# Original ConvNeXt Food-101 Training — Stage Extract

Cells **64–66** are copied verbatim from the team's historical research notebook, except the preceding Colab setup cell is provided to restore the original Food-101 90/10 seed-42 split. The original code trains the head before stage-3 fine-tuning and saves the historical checkpoint filenames. **This workflow has not been rerun as part of preparing this submission.** On a new run, stochastic training and library-version changes may produce different metrics.

Requires Colab GPU, storage and the original Food-101 dataset download. Requires adequate GPU RAM. Run cells in order.


In [ ]:
# Colab / Food-101 setup matching original experiment cell 4
%pip -q install -U timm tqdm
import os
from pathlib import Path
import torch
from torchvision.datasets import Food101
from torch.utils.data import random_split
from google.colab import drive

drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/NutriVision_AI')
save_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base = Food101(root='/content/data', split='train', download=True)
if len(base) != 75750:
    raise RuntimeError(f'Unexpected Food-101 training set size: {len(base)}')
split_file = save_dir / 'data_split.pth'
if split_file.exists():
    split = torch.load(split_file, map_location='cpu', weights_only=True)
else:
    val_size = int(len(base) * 0.10)
    train_ds, val_ds = random_split(base, [len(base)-val_size, val_size], generator=torch.Generator().manual_seed(42))
    split = {'train_indices': list(train_ds.indices), 'val_indices': list(val_ds.indices)}
    torch.save(split, split_file)
assert len(split['train_indices']) == 68175 and len(split['val_indices']) == 7575
assert not set(split['train_indices']).intersection(split['val_indices'])
print('Training:', len(split['train_indices']), 'Validation:', len(split['val_indices']), 'Device:', device)


## Initialize ConvNeXt and datasets (source cell 64)

In [ ]:

import torch
import timm

from torchvision import datasets, transforms as T
from torch.utils.data import DataLoader, Subset

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

save_dir = "/content/drive/MyDrive/NutriVision_AI"

# Keep the original Food-101 split
split = torch.load(
    f"{save_dir}/data_split.pth",
    weights_only=True
)

train_ids = split["train_indices"]
val_ids = split["val_indices"]

assert set(map(int, train_ids)).isdisjoint(
    set(map(int, val_ids))
), "Data leakage detected!"

# Strong pretrained classification backbone
model_name = "convnext_base.fb_in22k_ft_in1k_384"

fusion_cnn = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=101
)

# Read the model's correct preprocessing settings
data_cfg = timm.data.resolve_model_data_config(
    fusion_cnn
)

mean = data_cfg["mean"]
std = data_cfg["std"]

# Moderate augmentation for food photographs
fusion_train_transform = T.Compose([
    T.RandomResizedCrop(
        384,
        scale=(0.65, 1.0),
        interpolation=T.InterpolationMode.BICUBIC
    ),
    T.RandomHorizontalFlip(),
    T.RandAugment(num_ops=2, magnitude=7),
    T.ToTensor(),
    T.Normalize(mean, std)
])

# Deterministic validation preprocessing
fusion_val_transform = timm.data.create_transform(
    **data_cfg,
    is_training=False
)

train_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=fusion_train_transform,
    download=True
)

val_data = datasets.Food101(
    root="/content/data",
    split="train",
    transform=fusion_val_transform,
    download=False
)

assert train_data.classes == val_data.classes

fusion_train = Subset(train_data, train_ids)
fusion_val = Subset(val_data, val_ids)

fusion_train_loader = DataLoader(
    fusion_train,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

fusion_val_loader = DataLoader(
    fusion_val,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

fusion_cnn = fusion_cnn.to(device)

# Sanity check before expensive training
images, labels = next(iter(fusion_val_loader))
images = images.to(device)

fusion_cnn.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        enabled=(device.type == "cuda")
    ):
        outputs = fusion_cnn(images)

print("Model:", model_name)
print("Parameters:", sum(
    p.numel() for p in fusion_cnn.parameters()
) / 1e6, "million")
print("Images:", images.shape)
print("Outputs:", outputs.shape)
print("Training images:", len(fusion_train))
print("Validation images:", len(fusion_val))
print("Device:", device)


## Train classification head (source cell 65)

In [ ]:

import torch
import torch.nn as nn
from tqdm.auto import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Freeze the pretrained backbone
for param in fusion_cnn.parameters():
    param.requires_grad = False

# Train only the classification head
for param in fusion_cnn.head.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    fusion_cnn.head.parameters(),
    lr=5e-4,
    weight_decay=0.01
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_acc = -1.0

for epoch in range(1, 3):

    # Keep the frozen backbone in evaluation mode
    fusion_cnn.eval()
    fusion_cnn.head.train()

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for images, labels in tqdm(
        fusion_train_loader,
        desc=f"Training Epoch {epoch}"
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            enabled=use_amp
        ):
            outputs = fusion_cnn(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += batch_size

    # Validation
    fusion_cnn.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            fusion_val_loader,
            desc=f"Validation Epoch {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                enabled=use_amp
            ):
                outputs = fusion_cnn(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += batch_size

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    print(f"\nEpoch {epoch}/2")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": fusion_cnn.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "classes": train_data.classes,
            "model_name": model_name
        }, f"{save_dir}/convnext_head_best.pth")

        print("Best checkpoint saved!")

print(f"\nBest Head Accuracy: {best_acc:.2f}%")


## Fine-tune last ConvNeXt stage (source cell 66)

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 1. Restore the best head-training checkpoint
head_checkpoint = torch.load(
    f"{save_dir}/convnext_head_best.pth",
    map_location="cpu",
    weights_only=True
)

fusion_cnn.load_state_dict(
    head_checkpoint["model_state_dict"]
)
fusion_cnn = fusion_cnn.to(device)

print("Restored accuracy:", head_checkpoint["val_accuracy"])

# 2. Freeze all parameters
for param in fusion_cnn.parameters():
    param.requires_grad = False

# Unfreeze the last ConvNeXt stage
for param in fusion_cnn.stages[3].parameters():
    param.requires_grad = True

# Unfreeze the classification head
for param in fusion_cnn.head.parameters():
    param.requires_grad = True

# 3. Smaller batch size for GPU memory safety
fine_train_loader = DataLoader(
    fusion_train,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

fine_val_loader = DataLoader(
    fusion_val,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# 4. Different learning rates
optimizer = torch.optim.AdamW(
    [
        {
            "params": fusion_cnn.stages[3].parameters(),
            "lr": 1e-5
        },
        {
            "params": fusion_cnn.head.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

epochs = 3
accumulation_steps = 2

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-6
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_val_acc = head_checkpoint["val_accuracy"]

# 5. Train and validate
for epoch in range(1, epochs + 1):

    fusion_cnn.train()
    optimizer.zero_grad(set_to_none=True)

    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for step, (images, labels) in enumerate(
        tqdm(fine_train_loader, desc=f"Train {epoch}")
    ):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            outputs = fusion_cnn(images)
            loss = criterion(outputs, labels)

        # Gradient accumulation
        scaler.scale(
            loss / accumulation_steps
        ).backward()

        update_now = (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(fine_train_loader)
        )

        if update_now:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(
                    lambda p: p.requires_grad,
                    fusion_cnn.parameters()
                ),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = labels.size(0)

        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        train_total += batch_size
        train_loss_sum += loss.item() * batch_size

    # Validation
    fusion_cnn.eval()

    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(
            fine_val_loader,
            desc=f"Validation {epoch}"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                outputs = fusion_cnn(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()

            val_total += batch_size
            val_loss_sum += loss.item() * batch_size

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    scheduler.step()

    print(f"\nEpoch {epoch}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": fusion_cnn.state_dict(),
            "val_accuracy": val_acc,
            "train_accuracy": train_acc,
            "model_name": model_name,
            "classes": train_data.classes
        }, f"{save_dir}/convnext_stage3_best.pth")

        print("NEW BEST MODEL SAVED!")

    print(f"Best Accuracy: {best_val_acc:.2f}%")

print("\nFine-tuning finished!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
